In [1]:
import csv
import os
import pandas as pd
import numpy as np


In [2]:
def limpiar_y_reportar_docentes(
    ruta_sdemt: str,
    ruta_coe1: str,
    ruta_salida: str = "docentes_enoe_reporte_limpio.csv",
    reporte_salida: str = "reporte_limpieza.txt",
) -> pd.DataFrame:
  reporte = []
  reporte.append("==================================================")
  reporte.append("      REPORTE DE LIMPIEZA - DOCENTES ENOE         ")
  reporte.append("==================================================")

  # 1. Carga de archivos
  if not os.path.exists(ruta_sdemt) or not os.path.exists(ruta_coe1):
    raise FileNotFoundError(
        "No se encontraron los archivos de la ENOE en la ruta especificada."
    )

  reporte.append(f"Archivos de origen: {ruta_sdemt} y {ruta_coe1}")

  # Definir llaves de cruce
  llaves_cruce = ["cve_ent", "con", "upm", "v_sel", "n_hog", "h_mud", "n_ren"]

  # Columnas requeridas
  cols_sdemt = llaves_cruce + [
      "eda",
      "sex",
      "niv_ins",
      "cs_p13_1",  
      "cs_p13_2",
      "anios_esc",
      "hrsocup",
      "ingocup",
      "ing_x_hrs",
      "pos_ocu",
      "seg_soc",
      "clase1",
      "fac_tri",
  ]
  cols_coe1 = llaves_cruce + ["p3", "p3a", "p4", "p4a","p3c1","p3c2","p3c3"]

  df_sdemt = pd.read_csv(
      ruta_sdemt, usecols=cols_sdemt, encoding="latin1", low_memory=False
  )
  df_coe1 = pd.read_csv(
      ruta_coe1, usecols=cols_coe1, encoding="latin1", low_memory=False
  )

  reporte.append(f"Filas iniciales SDEMT: {len(df_sdemt)}")
  reporte.append(f"Filas iniciales COE1: {len(df_coe1)}")

  # 2. Cruce de DataFrames
  df_unido = pd.merge(df_sdemt, df_coe1, on=llaves_cruce, how="inner")
  reporte.append(
      f"[1] Registros totales tras la unión (Inner Join): {len(df_unido)}"
  )

  # 3. Filtrar personal docente (Códigos SINCO del grupo 23)
  df_docentes = df_unido[
      df_unido["p3"].astype(str).str.strip().str.startswith("23")
  ].copy()
  reporte.append(
      f"[2] Filtrado estricto de docentes (Grupo SINCO 23): {len(df_docentes)}"
  )

  # 4. Normalización de nombres de columnas
  cols_originales = list(df_docentes.columns)
  df_docentes.columns = (
      df_docentes.columns.astype(str)
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace(r"[^\w\s]", "", regex=True)
  )
  cambios_columnas = sum(
      1 for orig, nuev in zip(cols_originales, df_docentes.columns) if orig != nuev
  )
  reporte.append(f"[3] Columnas normalizadas: {cambios_columnas}")

  # 5. Estandarización de códigos especiales de INEGI ('\\N', espacios vacíos) a NaN
  df_docentes.replace([r"\\N", r"^\s*$", "NA", "NaN"], np.nan, regex=True, inplace=True)

  # 6. Conversión de tipos de datos principales a numéricos
  cols_numericas = ["eda", "sex", "anios_esc", "hrsocup", "ingocup", "fac_tri", "niv_ins", "cs_p13_1"]
  for col in cols_numericas:
    if col in df_docentes.columns:
      df_docentes[col] = pd.to_numeric(df_docentes[col], errors="coerce")

  # 7. Tratamiento inteligente de nulos (Filtrado de estructurales esenciales)
  filas_antes_nulos = len(df_docentes)
  cols_esenciales = ["eda", "sex", "anios_esc", "hrsocup", "ingocup", "fac_tri"]
  df_docentes.dropna(subset=cols_esenciales, inplace=True)
  reporte.append(
      f"[4] Filas eliminadas por valores nulos en variables esenciales"
      f" ({', '.join(cols_esenciales)}): {filas_antes_nulos - len(df_docentes)}"
  )

  # Tratamiento para complementarias (p.ej. contrato p4 -> rellenar nulos con 99 'No especificado')
  if "p4" in df_docentes.columns:
    df_docentes["p4"] = df_docentes["p4"].fillna(99)
    reporte.append(
        "[5] Valores nulos en 'p4' (contrato) rellenados con categoría 99 (No"
        " especificado)."
    )

  # 8. Depuración de valores lógicamente imposibles o absurdos
  filas_antes_atipicos = len(df_docentes)
  df_docentes = df_docentes[
      (df_docentes["eda"] >= 18)
      & (df_docentes["hrsocup"] > 0)
      & (df_docentes["hrsocup"] <= 98)
      & (df_docentes["ingocup"] > 0)
  ].copy()
  reporte.append(
      f"[6] Filas depuradas por valores extraños o fuera de rango (edad < 18,"
      f" horas 0/>98, ingresos <= 0): {filas_antes_atipicos - len(df_docentes)}"
  )

  # 9. Resumen Final
  reporte.append("\n==================================================")
  reporte.append("                  RESUMEN FINAL                   ")
  reporte.append("==================================================")
  reporte.append(f"Filas finales limpias: {len(df_docentes)}")
  reporte.append(f"Columnas finales: {len(df_docentes.columns)}")
  reporte.append(f"Archivo limpio guardado en: {ruta_salida}")

  texto_reporte = "\n".join(reporte)
  print("\n" + texto_reporte + "\n")

  # Guardar reporte en texto
  if reporte_salida:
    with open(reporte_salida, "w", encoding="utf-8") as f:
      f.write(texto_reporte)

  # Guardar CSV final optimizado para MySQL y análisis
  df_docentes.to_csv(ruta_salida, index=False, encoding="utf-8", na_rep="\\N")
  return df_docentes


# --- EJECUCIÓN DESDE EL NOTEBOOK ---
if __name__ == "__main__":
  ruta_sdemt = "data/ENOE_SDEMT226.csv"  # Ajusta tu ruta si es diferente
  ruta_coe1 = "data/ENOE_COE1T226.csv"  # Ajusta tu ruta si es diferente

  df_docentes_limpio = limpiar_y_reportar_docentes(
      ruta_sdemt=ruta_sdemt,
      ruta_coe1=ruta_coe1,
      ruta_salida="docentes_enoe_reporte_limpio.csv",
      reporte_salida="reporte_limpieza.txt",
  )


      REPORTE DE LIMPIEZA - DOCENTES ENOE         
Archivos de origen: data/ENOE_SDEMT226.csv y data/ENOE_COE1T226.csv
Filas iniciales SDEMT: 417128
Filas iniciales COE1: 344709
[1] Registros totales tras la unión (Inner Join): 369419
[2] Filtrado estricto de docentes (Grupo SINCO 23): 6857
[3] Columnas normalizadas: 0
[4] Filas eliminadas por valores nulos en variables esenciales (eda, sex, anios_esc, hrsocup, ingocup, fac_tri): 4
[5] Valores nulos en 'p4' (contrato) rellenados con categoría 99 (No especificado).
[6] Filas depuradas por valores extraños o fuera de rango (edad < 18, horas 0/>98, ingresos <= 0): 3401

                  RESUMEN FINAL                   
Filas finales limpias: 3452
Columnas finales: 27
Archivo limpio guardado en: docentes_enoe_reporte_limpio.csv



C:\Users\fersg\AppData\Local\Temp\ipykernel_42756\3204420791.py:91: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_docentes.dropna(subset=cols_esenciales, inplace=True)
